# 11 — PaliGemma Card Analysis via Ollama

Uses `jyan1/paligemma-mix-224` (Google PaliGemma, 224×224 input) running locally in Ollama  
to perform multi-task analysis on Pokémon / TCG card images.

## Tasks covered
| # | Task | PaliGemma prompt prefix |
|---|------|-------------------------|
| 1 | Card detection & bounding box | `detect pokemon card` |
| 2 | Card identification (name, set, rarity) | `caption en` / custom VQA |
| 3 | Condition assessment — corners, edges, surface | structured VQA |
| 4 | OCR (HP, card number, set symbol) | `ocr` |
| 5 | PSA grade prediction | chain-of-thought VQA |
| 6 | Visual QA probes | free-form questions |
| 7 | Batch analysis | all card images |
| 8 | Full grading report | front + back |
| 9 | Prompt variant comparison | 4 strategies |
| 10 | Interactive single-image | configure & run |

**Prerequisites:** `pip install ollama` · `ollama pull jyan1/paligemma-mix-224`

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME  = "jyan1/paligemma-mix-224"

# Default test images — change to any path you like
TEST_FRONT  = "cards/image0_front.jpeg"
TEST_BACK   = "cards/image0_back.jpeg"

# Print timing info per call
VERBOSE     = True

In [ ]:
# ── Imports & setup ───────────────────────────────────────────────────────────
import re, textwrap, time
from io import BytesIO
from pathlib import Path
from typing import Optional

import ollama
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from IPython.display import display, Markdown

# ── Verify model is available ─────────────────────────────────────────────────
try:
    available = [m.model for m in ollama.list().models]
    if MODEL_NAME not in available:
        print(f"⚠️  '{MODEL_NAME}' not found.")
        print(f"   Available models: {available}")
        print(f"   Run: ollama pull {MODEL_NAME}")
    else:
        print(f"✅  Model '{MODEL_NAME}' ready.")
except Exception as e:
    print(f"❌  Could not connect to Ollama: {e}")
    print("   Make sure Ollama is running: ollama serve")

## Helper — `paligemma()` wrapper

In [ ]:
# PaliGemma is a 224×224 model. Sending full-size photos (e.g. 3000×4000px)
# causes the Ollama runner to OOM and crash with status 500.
# We pre-resize to MAX_IMAGE_PX on the long side and hand bytes to ollama.
MAX_IMAGE_PX = 448   # 2× the native 224 — safe headroom, still enough detail


def prepare_image(path: str | Path) -> bytes:
    """
    Resize image so the long side ≤ MAX_IMAGE_PX, preserving aspect ratio.
    Returns JPEG bytes ready to pass to ollama.generate(images=[...]).
    """
    img = Image.open(path).convert("RGB")
    orig_w, orig_h = img.size
    img.thumbnail((MAX_IMAGE_PX, MAX_IMAGE_PX), Image.LANCZOS)
    new_w, new_h = img.size
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=90)
    if VERBOSE:
        print(f"  📐 resized {orig_w}×{orig_h} → {new_w}×{new_h}")
    return buf.getvalue()


def paligemma(
    prompt: str,
    image_path: str | Path,
    *,
    temperature: float = 0.0,
    max_tokens: int = 512,
) -> str:
    """
    Run a PaliGemma prompt + image through Ollama and return the text response.

    PaliGemma task prefixes (from the PaliGemma paper):
      'detect <thing>'   → <loc0NNN> bounding-box tokens  (y0 x0 y1 x1, 0-1023)
      'caption en'       → free-form English description
      'ocr'              → read all visible text
      '<question>'       → visual question answering

    Images are pre-resized to MAX_IMAGE_PX before sending to prevent
    the llama runner OOM crash (status 500) that occurs with full-size photos.
    """
    img_bytes = prepare_image(image_path)
    t0 = time.time()
    response = ollama.generate(
        model=MODEL_NAME,
        prompt=prompt,
        images=[img_bytes],
        options={"temperature": temperature, "num_predict": max_tokens},
    )
    elapsed = time.time() - t0
    text = response.response.strip()
    if VERBOSE:
        short = prompt[:55] + ("..." if len(prompt) > 55 else "")
        print(f"  ⏱  {elapsed:.1f}s | '{short}'")
    return text


def show_image(path: str | Path, title: str = "", ax=None):
    img = Image.open(path).convert("RGB")
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 7))
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    return img, ax


# Quick smoke-test
print("Smoke test — caption on front image")
cap = paligemma("caption en", TEST_FRONT)
print(f"Caption: {cap}")

## Task 1 — Card Detection (`detect` task)

PaliGemma encodes bounding boxes as `<loc0NNN>` tokens in `y0 x0 y1 x1` order  
(each value = coordinate / image_side × 1024, zero-padded to 4 digits).

In [ ]:
def parse_paligemma_boxes(raw: str, img_w: int, img_h: int) -> list[dict]:
    """
    Convert PaliGemma <loc0NNN> tokens into pixel bounding boxes.
    Tokens arrive in groups of 4: y0 x0 y1 x1 (normalised 0-1023).
    """
    token_re = re.compile(r'<loc(\d{4})>')
    tokens   = token_re.findall(raw)

    # Labels sit between box token groups
    label_re  = re.compile(r'>\s*([a-zA-Z][^<;\n]{0,40}?)(?=\s*(?:<loc|;|$))')
    labels    = [l.strip() for l in label_re.findall(raw)]

    boxes = []
    for i in range(0, len(tokens) - 3, 4):
        y0, x0, y1, x1 = [int(t) / 1024 for t in tokens[i:i+4]]
        boxes.append({
            "x0": int(x0 * img_w), "y0": int(y0 * img_h),
            "x1": int(x1 * img_w), "y1": int(y1 * img_h),
            "label": labels[i // 4] if i // 4 < len(labels) else "?",
        })
    return boxes


def detect_card(image_path: str | Path) -> dict:
    """Run PaliGemma detect task and draw bounding boxes."""
    img  = Image.open(image_path).convert("RGB")
    w, h = img.size

    raw = paligemma("detect pokemon card", image_path)
    print(f"  Raw: {raw!r}")

    boxes = parse_paligemma_boxes(raw, w, h)

    fig, ax = plt.subplots(figsize=(6, 8))
    ax.imshow(img)
    for b in boxes:
        rect = mpatches.Rectangle(
            (b["x0"], b["y0"]), b["x1"]-b["x0"], b["y1"]-b["y0"],
            linewidth=2, edgecolor="cyan", facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(b["x0"]+4, b["y0"]+14, b["label"], color="cyan",
                fontsize=9, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.5, pad=2, edgecolor="none"))
    ax.set_title(f"detect pokemon card  ({len(boxes)} box{'es' if len(boxes)!=1 else ''})",
                 fontsize=10)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    return {"raw": raw, "boxes": boxes}


det = detect_card(TEST_FRONT)

## Task 2 — Caption & Card Identification

In [ ]:
def identify_card(image_path: str | Path) -> dict:
    """Multi-prompt card identification: name, set, rarity, HP, card number."""
    prompts = {
        "caption":  "caption en",
        "name":     "What is the name of this Pokémon card?",
        "set":      "What Pokémon TCG set does this card belong to?",
        "rarity":   "What is the rarity of this card? (Common, Uncommon, Rare, Holo Rare, etc.)",
        "hp":       "What is the HP value printed on this card?",
        "card_num": "What is the card number and total printed on this card? (e.g. 4/102)",
    }
    results = {}
    print("Card Identification")
    print("═" * 50)
    for key, prompt in prompts.items():
        ans = paligemma(prompt, image_path, max_tokens=128)
        results[key] = ans
        print(f"  {key.replace('_',' ').title():<12}: {ans}")
    return results


ident = identify_card(TEST_FRONT)

## Task 3 — Condition Assessment

Targeted prompts for each PSA grading dimension.

In [ ]:
CONDITION_PROMPTS = {
    "corners": (
        "Examine the four corners of this trading card closely. "
        "Describe any whitening, fraying, or rounding you see. "
        "Rate each corner: Sharp / Minor Wear / Moderate Wear / Heavy Wear."
    ),
    "edges": (
        "Examine all four edges of this trading card. "
        "Describe any chipping, roughness, or whitening along the edges. "
        "Rate each edge: Clean / Minor Chips / Moderate Chips / Heavy Damage."
    ),
    "surface": (
        "Examine the surface of this trading card. "
        "Describe any scratches, print lines, dents, creases, or loss of gloss. "
        "Rate surface quality: Pristine / Minor Issues / Moderate Issues / Heavy Damage."
    ),
    "centering": (
        "Examine the centering of this trading card. "
        "Describe how evenly the image is centered within the card borders. "
        "Estimate left/right and top/bottom border ratios (e.g. 55/45 L/R)."
    ),
    "overall": (
        "You are a PSA card grading expert. "
        "Based on the visible condition of this trading card, "
        "provide an overall condition assessment and suggest a PSA grade from 1 to 10. "
        "Explain your reasoning briefly."
    ),
}


def assess_condition(
    front_path: str | Path,
    back_path: Optional[str | Path] = None,
) -> dict:
    results = {}
    print("Condition Assessment — Front")
    print("═" * 60)
    for dim, prompt in CONDITION_PROMPTS.items():
        ans = paligemma(prompt, front_path, max_tokens=256)
        results[f"front_{dim}"] = ans
        print(f"\n▸ {dim.upper()} (front)")
        for line in textwrap.wrap(ans, width=80):
            print(f"  {line}")

    if back_path:
        back_prompts = {
            "back_corners_edges": (
                "Examine the four corners and all four edges of the BACK of this trading card. "
                "Describe any whitening, fraying, or rounding. "
                "Rate overall back condition: Pristine / Minor Wear / Moderate Wear / Heavy Wear."
            ),
            "back_surface": (
                "Examine the back surface and centering of this trading card. "
                "Describe any scratches, print lines, or centering issues on the back."
            ),
        }
        print("\nCondition Assessment — Back")
        print("═" * 60)
        for name, prompt in back_prompts.items():
            ans = paligemma(prompt, back_path, max_tokens=256)
            results[name] = ans
            print(f"\n▸ {name.upper()}")
            for line in textwrap.wrap(ans, width=80):
                print(f"  {line}")
    return results


condition = assess_condition(TEST_FRONT, TEST_BACK)

## Task 4 — OCR

In [ ]:
def ocr_card(image_path: str | Path) -> str:
    """Run PaliGemma's native OCR task — reads all visible text."""
    raw = paligemma("ocr", image_path, max_tokens=512)
    print("OCR output:")
    print("-" * 50)
    print(raw)
    return raw


ocr_result = ocr_card(TEST_FRONT)

## Task 5 — PSA Grade Prediction (Chain-of-Thought)

In [ ]:
GRADE_PROMPT_FRONT = """\
You are a PSA-certified card grading expert evaluating a Pokémon trading card.

Analyze this card front image and score each dimension:

1. Centering (L/R and T/B ratios, max allowed: 55/45 for PSA 10)
2. Corners (all four: sharp vs worn)
3. Edges (all four: clean vs chipped)
4. Surface (scratches, print lines, gloss loss)

Then output:
CENTERING: <score 1-10 with brief rationale>
CORNERS: <score 1-10 with brief rationale>
EDGES: <score 1-10 with brief rationale>
SURFACE: <score 1-10 with brief rationale>
LIMITING FACTOR: <the weakest dimension>
PREDICTED PSA GRADE: <integer 1-10>
CONFIDENCE: <low | medium | high>
"""

GRADE_PROMPT_BACK = """\
You are a PSA-certified card grading expert evaluating the BACK of a Pokémon trading card.

Analyze corners, edges, surface, and centering on this back image.
Then output:
CORNERS: <score 1-10>
EDGES: <score 1-10>
SURFACE: <score 1-10>
CENTERING: <score 1-10>
BACK GRADE CONTRIBUTION: <integer 1-10>
"""


def grade_prediction(
    front_path: str | Path,
    back_path: Optional[str | Path] = None,
) -> dict:
    """Predict PSA grade using chain-of-thought prompts on front (+ optional back)."""
    print("PSA Grade Prediction")
    print("═" * 60)

    front_raw = paligemma(GRADE_PROMPT_FRONT, front_path, max_tokens=512)
    print("\n── FRONT ANALYSIS ──")
    print(front_raw)

    back_raw = None
    if back_path:
        back_raw = paligemma(GRADE_PROMPT_BACK, back_path, max_tokens=256)
        print("\n── BACK ANALYSIS ──")
        print(back_raw)

    grade_m = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', front_raw, re.I)
    conf_m  = re.search(r'CONFIDENCE[:\s]+(low|medium|high)', front_raw, re.I)
    grade   = float(grade_m.group(1)) if grade_m else None
    conf    = conf_m.group(1).lower() if conf_m else "unknown"

    print(f"\n{'─'*60}")
    if grade:
        print(f"  → Predicted PSA Grade: {grade:.0f}/10  {'⭐' * max(0, int(round(grade)))}")
        print(f"  → Confidence: {conf}")
    else:
        print("  → Could not parse grade from response.")

    return {"front_analysis": front_raw, "back_analysis": back_raw,
            "grade": grade, "confidence": conf}


grade_result = grade_prediction(TEST_FRONT, TEST_BACK)

## Task 6 — Visual QA Probes

In [ ]:
PROBE_QUESTIONS = [
    "Is there any visible print line on the front of the card?",
    "Are the card corners sharp or worn?",
    "Is this card properly centered or does it appear off-center?",
    "Does the card surface appear glossy or is the gloss worn?",
    "Is there any crease or bend visible on this card?",
    "What is the dominant color of the card border?",
    "Is this a holographic or non-holographic card?",
]

print("Visual QA Probes")
print("═" * 60)
probe_results = {}
for q in PROBE_QUESTIONS:
    ans = paligemma(q, TEST_FRONT, max_tokens=128)
    probe_results[q] = ans
    print(f"\nQ: {q}")
    print(f"A: {ans}")

## Task 7 — Batch Analysis Across All Card Images

In [ ]:
CARDS_DIR = Path("cards")
fronts = sorted(CARDS_DIR.glob("*_front.*"))
print(f"Found {len(fronts)} front image(s)")

batch_results = []
n = len(fronts)
fig, axes = plt.subplots(n, 2, figsize=(10, 5 * n)) if n > 1 else plt.subplots(1, 2, figsize=(10, 5))
axes = axes if n > 1 else [axes]

for i, front_path in enumerate(fronts):
    back_path = Path(str(front_path).replace("_front", "_back"))
    if not back_path.exists():
        back_path = None

    print(f"\n{'━'*60}")
    print(f"Card {i}: {front_path.name}")
    print(f"{'━'*60}")

    name_ans  = paligemma("What is the name of this Pokémon card?", front_path, max_tokens=64)
    grade_raw = paligemma(GRADE_PROMPT_FRONT, front_path, max_tokens=512)

    grade_m   = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', grade_raw, re.I)
    conf_m    = re.search(r'CONFIDENCE[:\s]+(low|medium|high)', grade_raw, re.I)
    lim_m     = re.search(r'LIMITING FACTOR[:\s]+([^\n]+)', grade_raw, re.I)

    grade    = float(grade_m.group(1)) if grade_m else None
    conf     = conf_m.group(1).lower() if conf_m else "?"
    limiting = lim_m.group(1).strip() if lim_m else "?"

    print(f"  Name:     {name_ans}")
    print(f"  Grade:    PSA {grade} (confidence: {conf})")
    print(f"  Limiting: {limiting}")

    grade_label = f"PSA {grade:.0f}" if grade else "PSA ?"
    axes[i][0].imshow(Image.open(front_path))
    axes[i][0].set_title(f"{name_ans}\n{grade_label} | conf: {conf}", fontsize=9)
    axes[i][0].axis("off")

    if back_path:
        axes[i][1].imshow(Image.open(back_path))
        axes[i][1].set_title("Back", fontsize=9)
    else:
        axes[i][1].text(0.5, 0.5, "No back image", ha="center", va="center", fontsize=10)
    axes[i][1].axis("off")

    batch_results.append(dict(
        path=str(front_path), name=name_ans, grade=grade,
        confidence=conf, limiting_factor=limiting, raw_analysis=grade_raw,
    ))

plt.suptitle("PaliGemma Batch Card Analysis", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Task 8 — Full Grading Report (Front + Back)

In [ ]:
def full_card_report(
    front_path: str | Path,
    back_path: Optional[str | Path] = None,
) -> dict:
    """Print a formatted PSA grading report for one card."""
    front_path = Path(front_path)

    card_name = paligemma("What is the name of this Pokémon card?", front_path, max_tokens=64)
    card_set  = paligemma("What Pokémon TCG set does this card belong to?", front_path, max_tokens=64)
    card_num  = paligemma("What is the card number and total?", front_path, max_tokens=32)
    grade_raw = paligemma(GRADE_PROMPT_FRONT, front_path, max_tokens=512)

    grade_m = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', grade_raw, re.I)
    conf_m  = re.search(r'CONFIDENCE[:\s]+(low|medium|high)', grade_raw, re.I)
    lim_m   = re.search(r'LIMITING FACTOR[:\s]+([^\n]+)', grade_raw, re.I)

    grade    = float(grade_m.group(1)) if grade_m else None
    conf     = conf_m.group(1) if conf_m else "?"
    limiting = lim_m.group(1).strip() if lim_m else "?"
    grade_str = f"PSA {grade:.0f}" if grade else "PSA ?"

    # Side-by-side image
    n_cols = 2 if back_path else 1
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 7))
    if n_cols == 1:
        axes = [axes]
    axes[0].imshow(Image.open(front_path))
    axes[0].set_title("Front", fontsize=9)
    axes[0].axis("off")
    if back_path:
        axes[1].imshow(Image.open(back_path))
        axes[1].set_title("Back", fontsize=9)
        axes[1].axis("off")
    fig.suptitle(f"{card_name} — {card_set} {card_num}\n{grade_str} | Confidence: {conf}",
                 fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f"""
╔══════════════════════════════════════════════════════════╗
║  PALIGEMMA GRADING REPORT
╠══════════════════════════════════════════════════════════╣
║  Card:           {card_name[:40]}
║  Set:            {card_set[:40]}
║  Number:         {card_num[:20]}
║  Predicted PSA:  {grade_str}
║  Confidence:     {conf}
║  Limiting:       {limiting[:40]}
╚══════════════════════════════════════════════════════════╝

── RAW ANALYSIS ──
{grade_raw}
""")
    return dict(card_name=card_name, grade=grade, confidence=conf,
                limiting=limiting, raw_analysis=grade_raw)


report = full_card_report(TEST_FRONT, TEST_BACK)

## Task 9 — Prompt Engineering Comparison

Four strategies for grade prediction on the same image.

In [ ]:
PROMPT_VARIANTS = {
    "simple": (
        "What PSA grade would you give this card? Answer with just a number from 1-10."
    ),
    "chain_of_thought": GRADE_PROMPT_FRONT,
    "expert_persona": (
        "You are a veteran PSA grader with 20 years of experience. "
        "Grade this card from 1-10 based on corners, edges, surface, and centering. "
        "Be precise and strict. Provide the grade as 'GRADE: X' at the end."
    ),
    "condition_first": (
        "First describe every visible defect on this trading card in detail. "
        "Then based on those defects, assign a PSA grade from 1-10. "
        "Format: DEFECTS: ... | GRADE: X"
    ),
}

print("Prompt Variant Comparison")
print("═" * 70)

variant_grades = {}
for name, prompt in PROMPT_VARIANTS.items():
    ans = paligemma(prompt, TEST_FRONT, max_tokens=512)
    g = re.search(r'(?:GRADE[:\s]+|PSA[:\s]+|PREDICTED PSA GRADE[:\s]+)([0-9\.]+)', ans, re.I)
    grade_found = float(g.group(1)) if g else None
    variant_grades[name] = grade_found
    print(f"\n▸ {name}  → PSA {grade_found}")
    print(f"  {ans[:200]}{'...' if len(ans)>200 else ''}")

print("\n" + "─"*40 + "\nSUMMARY\n" + "─"*40)
for name, g in variant_grades.items():
    print(f"  {name:<22} → PSA {g}")

## Task 10 — Interactive Single-Image Analysis

Change `ANALYZE_IMAGE`, `ANALYZE_BACK`, and `CUSTOM_QUESTION` to test any card.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
ANALYZE_IMAGE    = "cards/image1_front.jpeg"   # ← swap to any image path
ANALYZE_BACK     = "cards/image1_back.jpeg"    # ← or None
CUSTOM_QUESTION  = "Does this card have any print lines on the surface?"
# ─────────────────────────────────────────────────────────────────────────────

print(f"Analyzing: {ANALYZE_IMAGE}")
print("─" * 50)

cap        = paligemma("caption en", ANALYZE_IMAGE)
grade_raw  = paligemma(GRADE_PROMPT_FRONT, ANALYZE_IMAGE, max_tokens=512)
grade_m    = re.search(r'PREDICTED PSA GRADE[:\s]+([0-9\.]+)', grade_raw, re.I)
grade      = float(grade_m.group(1)) if grade_m else None
custom_ans = paligemma(CUSTOM_QUESTION, ANALYZE_IMAGE)

print(f"Caption:  {cap}")
print(f"Grade:    PSA {grade}")
print(f"Q: {CUSTOM_QUESTION}")
print(f"A: {custom_ans}")

back_arg = ANALYZE_BACK if ANALYZE_BACK and Path(ANALYZE_BACK).exists() else None
full_card_report(ANALYZE_IMAGE, back_arg)